# 01 — Formatos de configuração: o que cada um entrega (e o que esconde)

A mesma configuração escrita em **YAML, TOML, JSON, INI e `.env`** ([`configs/`](configs/)). O objetivo é ver, com execução:

1. como carregar cada formato só com a biblioteca padrão (+ PyYAML / python-dotenv);
2. **que tipos Python** cada formato entrega para o mesmo dado;
3. as pegadinhas de cada formato — e como uma camada de validação (dataclass/Pydantic, notebooks seguintes) as neutraliza.

> Regra que guia toda a pasta: **o arquivo é texto não confiável**. O formato decide quanto trabalho de conversão sobra; quem
> garante os tipos é a validação, nunca o parser.

In [1]:
import configparser
import json
import tomllib
from datetime import date
from pathlib import Path

import yaml
from dotenv import dotenv_values

CONFIGS = Path("configs")

## 1. Carregando cada formato

In [2]:
with open(CONFIGS / "app.json", encoding="utf-8") as f:
    from_json = json.load(f)

with open(CONFIGS / "app.toml", "rb") as f:          # tomllib exige modo binário
    from_toml = tomllib.load(f)

with open(CONFIGS / "app.yaml", encoding="utf-8") as f:
    from_yaml = yaml.safe_load(f)                     # NUNCA yaml.load sem Loader (executa tags arbitrárias)

ini = configparser.ConfigParser()
ini.read(CONFIGS / "app.ini", encoding="utf-8")
from_ini = {section: dict(ini[section]) for section in ini.sections()}

from_env = dotenv_values(CONFIGS / "app.env")         # não mexe em os.environ

print(f"from_json['server']: {from_json['server']}")
print(f"from_toml['server']: {from_toml['server']}")
print(f"from_yaml['server']: {from_yaml['server']}")
print(f"from_ini['server']: {from_ini['server']}")
print(f"from_env['SERVER__PORT']: {from_env['SERVER__PORT']!r}")

from_json['server']: {'host': '0.0.0.0', 'port': 8080, 'timeout_s': 2.5}
from_toml['server']: {'host': '0.0.0.0', 'port': 8080, 'timeout_s': 2.5}
from_yaml['server']: {'host': '0.0.0.0', 'port': 8080, 'timeout_s': 2.5}
from_ini['server']: {'host': '0.0.0.0', 'port': '8080', 'timeout_s': '2.5', 'log_level': 'info'}
from_env['SERVER__PORT']: '8080'


## 2. O mesmo dado, tipos diferentes

Mesmos valores, cinco formatos. Que tipo Python chega para cada um?

In [3]:
def pick(source: str, key: str):
    match source:
        case "json":
            data = from_json
        case "toml":
            data = from_toml
        case "yaml":
            data = from_yaml
        case "ini":
            data = {k: v for section in from_ini.values() for k, v in section.items()}
            return data.get(key.split(".")[-1])
        case "env":
            return from_env.get(key.upper().replace(".", "__"))
    for part in key.split("."):
        data = data[part]
    return data


KEYS = ["server.port", "server.timeout_s", "app.debug", "release_date", "tags"]
print(f"{'chave':<18}" + "".join(f"{fmt:<12}" for fmt in ["json", "toml", "yaml", "ini", "env"]))
for key in KEYS:
    row = "".join(f"{type(pick(fmt, key)).__name__:<12}" for fmt in ["json", "toml", "yaml", "ini", "env"])
    print(f"{key:<18}{row}")

chave             json        toml        yaml        ini         env         
server.port       int         int         int         str         str         
server.timeout_s  float       float       float       str         str         
app.debug         bool        bool        bool        str         str         
release_date      str         date        date        str         str         
tags              list        list        list        str         str         


Leitura da tabela:

| formato | tipos nativos | datas | listas/aninhamento | `null` | comentários |
|---|---|---|---|---|---|
| JSON | str, int, float, bool, list, dict | ❌ (string) | ✅ | ✅ | ❌ |
| TOML | + `date`/`datetime` | ✅ | ✅ | **❌ não existe** | ✅ |
| YAML | + `date`, inferência implícita | ✅ | ✅ | ✅ | ✅ |
| INI | **só str** | ❌ | só seções | ❌ | ✅ |
| `.env` | **só str** | ❌ | por convenção (`A__B`) | ❌ | ✅ |

Quanto menos tipos o formato tem, mais a camada de validação precisa **converter**. Quanto mais ele infere (YAML), mais ela precisa
**desconfiar**.

## 3. YAML: inferência implícita (o "problema da Noruega")

O PyYAML segue o YAML 1.1, em que vários escalares sem aspas viram outra coisa:

In [4]:
surprise = yaml.safe_load("""
country: NO          # código da Noruega
mode: 010            # permissão de arquivo
duration: 3:30       # três minutos e meio
version: 1.10        # versão
flag: on
empty:
""")
for key, value in surprise.items():
    print(f"{key}: {value!r}")

country: False
mode: 8
duration: 210
version: 1.1
flag: True
empty: None


`NO` virou `False`, `010` virou `8` (octal), `3:30` virou `210` (sexagesimal), `1.10` virou `1.1`. Nenhum erro, nenhum aviso.

Duas defesas, em camadas:
- **no arquivo:** aspas em tudo o que é texto (`country: "NO"`);
- **no código:** um modelo que exige `str`. O Pydantic **não** converte `bool`/`int` em `str`, então o erro aparece na carga:

In [5]:
from pydantic import BaseModel, ValidationError


class Settings(BaseModel):
    country: str
    mode: str
    duration: str
    version: str


try:
    Settings.model_validate(surprise)
except ValidationError as ex:
    for error in ex.errors():
        print(f"{error['loc'][0]}: {error['msg']} (veio {error['input']!r})")

country: Input should be a valid string (veio False)
mode: Input should be a valid string (veio 8)
duration: Input should be a valid string (veio 210)
version: Input should be a valid string (veio 1.1)


> ⚠️ O modelo pegou os quatro, inclusive `version` (`float` também não vira `str`). Mas veja o valor na mensagem: `1.1`, não
> `1.10`. A informação original já foi perdida no parser; a validação só consegue dizer que **algo** está errado. Aspas no arquivo
> continuam necessárias.

## 4. INI: tudo é string, chaves sem caixa, interpolação com `%`

In [6]:
parser = configparser.ConfigParser()
parser.read_string("""
[DEFAULT]
timeout = 5

[db]
Host = localhost
password = abc%123
""")

print(f"parser['db']['HOST']: {parser['db']['HOST']}")                 # chaves são normalizadas para minúsculas
print(f"list(parser['db']): {list(parser['db'])}")                   # DEFAULT "vaza" para todas as seções
print(f"parser['db'].getint('timeout'): {parser['db'].getint('timeout')}")
print(f"ConfigParser.BOOLEAN_STATES: {configparser.ConfigParser.BOOLEAN_STATES}")
try:
    parser["db"]["password"]
except configparser.InterpolationSyntaxError as ex:
    print(f"{type(ex).__name__}: {ex}")

raw = configparser.ConfigParser(interpolation=None)                   # desliga a interpolação
raw.read_string("[db]\npassword = abc%123\n")
print(f"raw['db']['password']: {raw['db']['password']}")

parser['db']['HOST']: localhost
list(parser['db']): ['host', 'password', 'timeout']
parser['db'].getint('timeout'): 5
ConfigParser.BOOLEAN_STATES: {'1': True, 'yes': True, 'true': True, 'on': True, '0': False, 'no': False, 'false': False, 'off': False}
InterpolationSyntaxError: '%' must be followed by '%' or '(', found: '%123'
raw['db']['password']: abc%123


- Uma senha com `%` **quebra** a leitura com o `ConfigParser` padrão: use `interpolation=None` (ou `RawConfigParser`).
- A seção `DEFAULT` entra em **todas** as seções — útil, mas surpreende quem itera pelas chaves.
- `getboolean` aceita `yes/no/on/off/1/0/true/false`; o Pydantic aceita um conjunto parecido para `bool`, então dá para deixar a
  conversão para o modelo.

## 5. TOML: tipado, mas sem `null`

In [7]:
print(f"from_toml['release_date']: {from_toml['release_date']!r}")
try:
    tomllib.loads("proxy = null")
except tomllib.TOMLDecodeError as ex:
    print(f"TOMLDecodeError: {ex}")

from_toml['release_date']: datetime.date(2026, 1, 15)
TOMLDecodeError: Invalid value (at line 1, column 9)


TOML não tem `null`: "sem valor" se expressa **omitindo a chave** — e o modelo precisa de um default para o campo ausente.
O `tomllib` só lê; para escrever é preciso um pacote extra (`tomli-w`).

## 6. JSON: sem datas e sem comentários

In [8]:
print(f"from_json['release_date']: {from_json['release_date']!r}")
try:
    json.loads('{"port": 8080, // porta\n}')
except json.JSONDecodeError as ex:
    print(f"JSONDecodeError: {ex}")

from_json['release_date']: '2026-01-15'
JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 16 (char 15)


## 7. `.env` e variáveis de ambiente: plano e só string

In [9]:
for key in ["SERVER__PORT", "APP__DEBUG", "TAGS"]:
    print(f"from_env[{key!r}]: {from_env[key]!r}")

from_env['SERVER__PORT']: '8080'
from_env['APP__DEBUG']: 'true'
from_env['TAGS']: '["api", "interno"]'


Tudo string, inclusive a "lista" `TAGS`, que só vira lista se alguém fizer o parse do JSON (o `pydantic-settings` faz isso para
campos complexos — notebook [03](03_configuracao_tipada.ipynb)). O aninhamento depende de uma convenção (`SERVER__PORT`).

## 8. Uma validação para todos os formatos

O ponto central: depois da validação, **o formato deixa de importar**. Um único modelo recebe os cinco dicionários (o INI e o `.env`
precisam só de um ajuste de forma, não de tipo):

In [10]:
from datetime import date
from enum import StrEnum
from pydantic import BaseModel, ConfigDict, Field


class LogLevel(StrEnum):
    DEBUG = "debug"
    INFO = "info"
    WARNING = "warning"


class AppSection(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: str
    environment: str
    debug: bool


class ServerSection(BaseModel):
    model_config = ConfigDict(extra="forbid")
    host: str
    port: int = Field(ge=1, le=65535)
    timeout_s: float = Field(gt=0)


class DatabaseSection(BaseModel):
    model_config = ConfigDict(extra="forbid")
    url: str
    pool_size: int = Field(ge=1)


class AppConfig(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)
    app: AppSection
    server: ServerSection
    database: DatabaseSection
    release_date: date
    tags: list[str]
    log_level: LogLevel


def ini_to_shape(sections: dict[str, dict[str, str]]) -> dict:
    """INI só tem seções de strings: reorganiza (forma), sem converter tipos."""
    app = dict(sections["app"])
    return {
        "app": {k: app[k] for k in ("name", "environment", "debug")},
        "server": {k: v for k, v in sections["server"].items() if k != "log_level"},
        "database": {k: v for k, v in sections["database"].items() if k != "log_level"},
        "release_date": app["release_date"],
        "tags": [t.strip() for t in app["tags"].split(",")],
        "log_level": app["log_level"],
    }


def env_to_shape(env: dict[str, str]) -> dict:
    """.env é plano: A__B vira {'a': {'b': ...}}; TAGS é JSON."""
    nested: dict = {}
    for key, value in env.items():
        *parents, leaf = key.lower().split("__")
        node = nested
        for parent in parents:
            node = node.setdefault(parent, {})
        node[leaf] = json.loads(value) if key == "TAGS" else value
    return nested


loaded = {
    "json": AppConfig.model_validate(from_json),
    "toml": AppConfig.model_validate(from_toml),
    "yaml": AppConfig.model_validate(from_yaml),
    "ini": AppConfig.model_validate(ini_to_shape(from_ini)),
    "env": AppConfig.model_validate(env_to_shape(from_env)),
}
first = loaded["json"]
print(f"todos iguais: {all(cfg == first for cfg in loaded.values())}")
print(f"loaded['ini'].server: {loaded['ini'].server!r}")
print(f"loaded['env'].release_date: {loaded['env'].release_date!r}")

todos iguais: True
loaded['ini'].server: ServerSection(host='0.0.0.0', port=8080, timeout_s=2.5)
loaded['env'].release_date: datetime.date(2026, 1, 15)


Cinco formatos, objetos iguais (`==` compara campo a campo). As strings do INI e do `.env`
foram convertidas pelo modelo; as datas do YAML/TOML entraram como `date`; as do JSON, como texto ISO convertido.

## 9. Qual formato escolher?

| precisa de… | escolha | por quê |
|---|---|---|
| config de aplicação/projeto editada por humanos | **TOML** | tipado, sem inferência mágica, padrão do ecossistema (`pyproject.toml`), `tomllib` na stdlib |
| estruturas profundas, listas de objetos, âncoras | **YAML** (com `safe_load` + validação) | expressivo; exige aspas e validação estrita |
| troca entre programas | **JSON** | universal; sem comentários |
| legado / ferramentas antigas | **INI** | só se já existir |
| segredos e ajustes por ambiente | **variáveis de ambiente / `.env`** | 12-factor; nunca no repositório |

In [11]:
assert loaded["yaml"] == loaded["ini"] == loaded["env"]
assert isinstance(loaded["json"].release_date, date)
print("verificações: ok")

verificações: ok
